# SSPNet 추가 학습 · 무료 Colab

YoloBeachTrain v2의 train·valid·test 전체 2,961장을 학습에 사용하고 `person water`와 `person sand`를 `person`으로 합칩니다. Piriapolis는 학습과 가중치 선택에 사용하지 않습니다.

무료 GPU를 선택하고 1→2→3→4→5 순서로 실행하세요. Drive 인증은 직접 완료합니다. 먼저 시험 학습의 시간과 메모리를 확인하세요. 본 학습 횟수는 Piriapolis 결과를 보기 전에 고정하며 최종 epoch 가중치를 평가에 사용합니다. 기본 5 epoch는 시작 제안이며 성능을 보장하지 않습니다.

중단되면 같은 설정·실행 이름으로 1→2→3을 실행하고 5단계의 `RESUME`을 켭니다. epoch가 끝날 때마다 optimizer를 포함한 체크포인트를 Drive에 저장하므로 마지막으로 완료한 epoch부터 재개합니다. 시험 학습은 본 학습 가중치에 반영되지 않습니다. 데이터에 이미 포함된 증강 외에 추가 증강은 적용하지 않습니다.


In [ ]:
#@title 1. 설정
import os, sys, json, shutil, subprocess
from pathlib import Path
EPOCHS = 5 #@param {type:"integer"}
SMOKE_STEPS = 50 #@param {type:"integer"}
RUN_NAME = "beach_v2_5epochs" #@param {type:"string"}
DATASET_ZIP_PATH = "" #@param {type:"string"}
if EPOCHS < 1 or SMOKE_STEPS < 1 or not RUN_NAME or any(c not in 'abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789_-' for c in RUN_NAME):
    raise ValueError('횟수와 영문·숫자·밑줄·하이픈으로 된 실행 이름을 확인하세요.')
WORK=Path('/content/sspnet_free')
WORK.mkdir(exist_ok=True)
print('설정:', RUN_NAME, EPOCHS, 'epoch / 시험', SMOKE_STEPS, '회')


In [ ]:
#@title 2. 환경·공개 가중치 준비
HELPER_SOURCES = {'setup_environment.py': '"""무료 Colab의 기본 Python과 분리하여 SSPNet 실행 환경을 만든다."""\nimport json\nimport os\nimport platform\nimport shutil\nimport subprocess\nimport sys\nfrom pathlib import Path\n\nROOT = Path("/content/sspnet_free")\nREPO = ROOT / "SSPNet"\nVENV = ROOT / "venv"\nPYTHON = VENV / "bin/python"\nCOMMIT = "f2d0f42273945577a5c6497fd2b384cb545f8f66"\nSTAMP = "python38-torch110-cu111-mmcv1318-v1"\n\n\ndef command(*args, env=None):\n    subprocess.run([str(a) for a in args], check=True, env=env)\n\n\ndef setup():\n    if platform.system() != "Linux" or platform.machine() != "x86_64" or not Path("/content").is_dir():\n        raise RuntimeError("이 설치 셀은 Google Colab Linux 런타임에서 실행하세요.")\n    if not shutil.which("nvidia-smi"):\n        raise RuntimeError("무료 Colab에서 런타임 유형 변경 → GPU를 선택한 뒤 다시 실행하세요.")\n    command("nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv")\n    free = shutil.disk_usage("/content").free / 1024**3\n    print("Colab 임시 디스크 여유: %.1f GiB" % free, flush=True)\n    if free < 8:\n        raise RuntimeError("환경 설치를 시작하려면 임시 디스크에 최소 8 GiB 여유가 필요합니다.")\n    ROOT.mkdir(exist_ok=True)\n    if not PYTHON.is_file():\n        command(sys.executable,"-m","pip","install","--quiet","--no-cache-dir","uv==0.9.7")\n        command("uv","python","install","3.8.20")\n        command("uv","venv","--python","3.8.20","--seed",VENV)\n    env = os.environ.copy()\n    env.pop("PYTHONPATH",None)\n    env.pop("PYTHONHOME",None)\n    env["PIP_DISABLE_PIP_VERSION_CHECK"] = "1"\n    env["PYTHONNOUSERSITE"] = "1"\n    marker = ROOT / "environment_ready.json"\n    ready = json.loads(marker.read_text()).get("stamp") == STAMP if marker.exists() else False\n    if not ready:\n        command(PYTHON,"-m","pip","install","--no-cache-dir","pip==24.3.1",\n                "setuptools==59.5.0","wheel==0.45.1",env=env)\n        command(PYTHON,"-m","pip","install","--no-cache-dir",\n                "torch==1.10.0+cu111","torchvision==0.11.1+cu111",\n                "--find-links","https://download.pytorch.org/whl/cu111/torch_stable.html",env=env)\n        command(PYTHON,"-m","pip","install","--no-cache-dir",\n                "numpy==1.21.6","scipy==1.7.3","matplotlib==3.5.3","Pillow==9.5.0",\n                "opencv-python-headless==4.5.5.64","pycocotools==2.0.7",\n                "addict==2.4.0","yapf==0.31.0","PyYAML==6.0.1","packaging==23.2",\n                "terminaltables==3.1.10","six==1.16.0","tqdm==4.66.5","gdown==5.2.0",env=env)\n        # 사전 빌드된 연산자만 설치한다. 무료 GPU 시간으로 CUDA 소스 빌드를 하지 않는다.\n        command(PYTHON,"-m","pip","install","--no-cache-dir","--no-deps",\n                "https://download.openmmlab.com/mmcv/dist/cu111/torch1.10.0/"\n                "mmcv_full-1.3.18-cp38-cp38-manylinux1_x86_64.whl",env=env)\n    if not REPO.exists():\n        command("git","init",REPO)\n        command("git","-C",REPO,"remote","add","origin","https://github.com/MingboHong/SSPNet.git")\n        command("git","-C",REPO,"fetch","--depth=1","origin",COMMIT)\n        command("git","-C",REPO,"checkout","--detach","FETCH_HEAD")\n    actual = subprocess.check_output(["git","-C",str(REPO),"rev-parse","HEAD"],text=True).strip()\n    if actual != COMMIT:\n        raise RuntimeError("기존 SSPNet 코드 버전이 다릅니다. 새 무료 런타임에서 설치 셀부터 실행하세요.")\n    env["PYTHONPATH"] = str(REPO)\n    check = \'\'\'\nimport json, sys, torch, torchvision, mmcv, mmdet\nfrom mmcv.ops import nms, roi_align\nassert torch.__version__ == \'1.10.0+cu111\'\nassert mmcv.__version__ == \'1.3.18\'\nassert torch.cuda.is_available(), \'GPU를 사용할 수 없습니다.\'\na = torch.randn(32,32,device=\'cuda\'); (a @ a).sum().item()\nb = torch.tensor([[0.,0.,10.,10.],[1.,1.,9.,9.]],device=\'cuda\')\ns = torch.tensor([.9,.8],device=\'cuda\')\nnms(b,s,.5)\nroi_align(torch.ones(1,1,16,16,device=\'cuda\'),torch.tensor([[0.,1.,1.,8.,8.]],device=\'cuda\'),(3,3))\ntorch.cuda.synchronize()\nprint(json.dumps(dict(python=sys.version,torch=torch.__version__,torchvision=torchvision.__version__,\n    cuda=torch.version.cuda,mmcv=mmcv.__version__,mmdet=mmdet.__version__,\n    gpu=torch.cuda.get_device_name(0),capability=torch.cuda.get_device_capability(0)),ensure_ascii=False,indent=2))\n\'\'\'\n    command(PYTHON,"-c",check,env=env)\n    marker.write_text(json.dumps(dict(stamp=STAMP,source_commit=actual)),encoding="utf-8")\n    freeze = subprocess.check_output([str(PYTHON),"-m","pip","freeze"],env=env,text=True)\n    (ROOT/"installed_packages.txt").write_text(freeze,encoding="utf-8")\n    print("환경 확인 완료: CUDA 행렬 연산, NMS, RoIAlign, SSPNet 패키지 가져오기 성공",flush=True)\n\n\nif __name__ == "__main__":\n    setup()\n', 'metrics.py': '"""원본 이미지 좌표에서 수영자 탐지와 크기별 재현율을 평가한다."""\nimport json\nimport math\nimport re\nimport os\nfrom collections import Counter, defaultdict\nfrom pathlib import Path\n\n\ndef canonical(value):\n    return " ".join(re.sub(r"[_-]+", " ", value.lower()).split())\n\n\ndef image_path(annotation_path, name):\n    base = Path(annotation_path).resolve().parent\n    path = (base / name).resolve()\n    if base != path and base not in path.parents:\n        raise ValueError("이미지 경로가 데이터셋 폴더 밖을 가리킵니다: " + name)\n    if not path.is_file():\n        raise FileNotFoundError("정답 파일과 같은 폴더 아래에 이미지가 필요합니다: " + str(path))\n    return path\n\n\ndef size_group(box):\n    d = max(float(box[2]), float(box[3]))\n    return "0<d<=8" if d <= 8 else "8<d<=12" if d <= 12 else "12<d<=20" if d <= 20 else "20<d"\n\n\ndef geometric_size(box):\n    return math.sqrt(float(box[2]) * float(box[3]))\n\n\ndef merge_splits(annotation_paths, output_path):\n    """분할별로 겹치는 ID와 파일명을 구분하고 이미지 복사 없이 합친다."""\n    output_path = Path(output_path).resolve()\n    base = output_path.parent\n    merged = dict(info={"description": "전체 분할 통합", "source_splits": []},\n                  licenses=[], images=[], annotations=[], categories=[])\n    categories = {}\n    seen_paths = set()\n    for split, source in annotation_paths:\n        source = Path(source).resolve()\n        if split in merged["info"]["source_splits"]:\n            raise ValueError("같은 분할을 두 번 합칠 수 없습니다: " + split)\n        raw = json.loads(source.read_text(encoding="utf-8"))\n        merged["info"]["source_splits"].append(split)\n        catmap = {}\n        for category in raw["categories"]:\n            name = canonical(category["name"])\n            if name not in categories:\n                categories[name] = len(categories) + 1\n                merged["categories"].append(dict(id=categories[name], name=category["name"]))\n            catmap[category["id"]] = categories[name]\n        images = {}\n        for im in sorted(raw["images"], key=lambda v: v["id"]):\n            if im["id"] in images:\n                raise ValueError("분할 안에서 이미지 ID가 중복됩니다: " + split)\n            path = image_path(source, im["file_name"])\n            if path in seen_paths:\n                raise ValueError("같은 이미지 경로가 중복됩니다: " + str(path))\n            seen_paths.add(path)\n            if base not in path.parents:\n                raise ValueError("통합 정답은 모든 이미지의 상위 폴더에 저장해야 합니다.")\n            new_id = len(merged["images"])\n            images[im["id"]] = new_id\n            merged["images"].append(dict(im, id=new_id, file_name=path.relative_to(base).as_posix(),\n                                         source_split=split, source_image_id=im["id"]))\n        seen_annotations = set()\n        for a in raw["annotations"]:\n            if a["id"] in seen_annotations:\n                raise ValueError("분할 안에서 정답 ID가 중복됩니다: " + split)\n            seen_annotations.add(a["id"])\n            if a["image_id"] not in images or a["category_id"] not in catmap:\n                raise ValueError("정답의 이미지 또는 종류 ID를 찾지 못했습니다: " + split)\n            merged["annotations"].append(dict(a, id=len(merged["annotations"])+1,\n                image_id=images[a["image_id"]], category_id=catmap[a["category_id"]],\n                source_split=split, source_annotation_id=a["id"]))\n    if not merged["images"]:\n        raise ValueError("합칠 이미지가 없습니다.")\n    output_path.parent.mkdir(parents=True, exist_ok=True)\n    temporary = output_path.with_suffix(".json.tmp")\n    temporary.write_text(json.dumps(merged, ensure_ascii=False, indent=2), encoding="utf-8")\n    os.replace(str(temporary), str(output_path))\n    return output_path\n\n\ndef load_dataset(annotation_path, target_names=("person water",), limit=0):\n    raw = json.loads(Path(annotation_path).read_text(encoding="utf-8"))\n    cats = raw.get("categories", [])\n    wanted = {canonical(s) for s in target_names}\n    selected = [c["id"] for c in cats if canonical(c["name"]) in wanted]\n    if not selected:\n        raise ValueError("물속 사람 라벨을 찾지 못했습니다. 실제 종류: " + str([c["name"] for c in cats]))\n    images = sorted(raw["images"], key=lambda im: im["id"])\n    if len({im["id"] for im in images}) != len(images):\n        raise ValueError("이미지 ID가 중복되어 있습니다.")\n    if limit < 0:\n        raise ValueError("이미지 제한은 0 이상이어야 합니다.")\n    if limit:\n        images = images[:limit]\n    if not images:\n        raise ValueError("선택한 분할에 이미지가 없습니다.")\n    imap = {im["id"]: im for im in images}\n    ann_ids = set()\n    annotations = []\n    for a in raw["annotations"]:\n        if a["id"] in ann_ids:\n            raise ValueError("정답 ID가 중복되어 있습니다.")\n        ann_ids.add(a["id"])\n        if a["image_id"] not in imap or a["category_id"] not in selected:\n            continue\n        x, y, w, h = map(float, a["bbox"])\n        im = imap[a["image_id"]]\n        if not all(math.isfinite(v) for v in (x, y, w, h)) or w <= 0 or h <= 0:\n            raise ValueError("유효하지 않은 정답 박스: " + str(a["id"]))\n        if x < -.5 or y < -.5 or x + w > im["width"] + .5 or y + h > im["height"] + .5:\n            raise ValueError("이미지 밖의 정답 박스입니다. 좌표 변환을 확인하세요: " + str(a["id"]))\n        annotations.append(dict(id=a["id"], image_id=a["image_id"], category_id=1,\n                                bbox=[x, y, w, h], area=w*h,\n                                iscrowd=int(a.get("iscrowd", 0)),\n                                ignore=bool(a.get("ignore", False)), segmentation=[]))\n    for im in images:\n        path = image_path(annotation_path, im["file_name"])\n        from PIL import Image\n        with Image.open(path) as pic:\n            if pic.size != (im["width"], im["height"]):\n                raise ValueError("이미지 크기와 정답 메타데이터가 다릅니다: " + str(path))\n    gt = dict(info={"description": "물속 사람 평가용 정답"}, licenses=[], images=images,\n              annotations=annotations, categories=[dict(id=1, name="person water")])\n    counts = Counter(size_group(a["bbox"]) for a in annotations if not a["ignore"] and not a["iscrowd"])\n    normal = [a for a in annotations if not a["ignore"] and not a["iscrowd"]]\n    split_images = Counter(im.get("source_split", "single") for im in images)\n    image_splits = {im["id"]: im.get("source_split", "single") for im in images}\n    split_targets = Counter(image_splits[a["image_id"]] for a in normal)\n    return gt, dict(image_count=len(images), available_image_count=len(raw["images"]), target_count=sum(counts.values()),\n                    geometric_2to20_count=sum(2 <= geometric_size(a["bbox"]) < 20 for a in normal),\n                    geometric_2to8_count=sum(2 <= geometric_size(a["bbox"]) < 8 for a in normal),\n                    split_counts={s: dict(images=n, water_targets=split_targets[s]) for s,n in split_images.items()},\n                    size_counts=dict(counts), available_classes=[c["name"] for c in cats],\n                    target_classes=[c["name"] for c in cats if c["id"] in selected])\n\n\ndef overlap(a, b, crowd=False):\n    x, y, w, h = a\n    u, v, s, t = b\n    inter = max(0., min(x+w, u+s)-max(x, u)) * max(0., min(y+h, v+t)-max(y, v))\n    area = w*h if crowd else w*h+s*t-inter\n    return inter / area if area > 0 else 0.\n\n\ndef match_predictions(gt, predictions, iou=.5):\n    normal, ignored = defaultdict(list), defaultdict(list)\n    for a in gt["annotations"]:\n        (ignored if a.get("ignore") or a.get("iscrowd") else normal)[a["image_id"]].append(a)\n    used = set()\n    valid_ids = {im["id"] for im in gt["images"]}\n    ranked = sorted((p for p in predictions if p["image_id"] in valid_ids), key=lambda p: -p["score"])\n    matches = []\n    for p in ranked:\n        candidates = [(overlap(p["bbox"], a["bbox"]), a) for a in normal[p["image_id"]] if a["id"] not in used]\n        score, best = max(candidates, key=lambda pair: pair[0]) if candidates else (0., None)\n        if best is not None and score >= iou:\n            used.add(best["id"])\n            matches.append((p, best, False))\n        elif any(overlap(p["bbox"], a["bbox"], bool(a.get("iscrowd"))) >= iou for a in ignored[p["image_id"]]):\n            matches.append((p, None, True))\n        else:\n            matches.append((p, None, False))\n    return matches\n\n\ndef evaluate(gt, predictions, score_threshold=.25, iou=.5):\n    matches = match_predictions(gt, predictions, iou)\n    counts = Counter(size_group(a["bbox"]) for a in gt["annotations"] if not a.get("ignore") and not a.get("iscrowd"))\n    positives = sum(counts.values())\n    hits, tp, fp = Counter(), 0, 0\n    recalls, precisions = [], []\n    total_tp = total_fp = 0\n    for p, a, ignored in matches:\n        if ignored:\n            continue\n        total_tp += a is not None\n        total_fp += a is None\n        recalls.append(total_tp / positives if positives else 0.)\n        precisions.append(total_tp / (total_tp + total_fp))\n        if p["score"] >= score_threshold:\n            tp += a is not None\n            fp += a is None\n            if a is not None:\n                hits[size_group(a["bbox"])] += 1\n    ap = sum(max((p for r, p in zip(recalls, precisions) if r >= q/100), default=0.) for q in range(101))/101 if positives else None\n    tiny_count = sum(counts[k] for k in ("0<d<=8", "8<d<=12", "12<d<=20"))\n    tiny_hits = sum(hits[k] for k in ("0<d<=8", "8<d<=12", "12<d<=20"))\n    def geometric_range(low, high):\n        selected = {a["id"] for a in gt["annotations"] if not a.get("ignore") and not a.get("iscrowd")\n                    and low <= geometric_size(a["bbox"]) < high}\n        found = sum(a is not None and a["id"] in selected for p,a,ignored in matches\n                    if not ignored and p["score"] >= score_threshold)\n        return dict(positives=len(selected), hits=found, recall=found/len(selected) if selected else None)\n    return dict(iou=iou, score_threshold=score_threshold, images=len(gt["images"]),\n                positives=positives, true_positives=tp, false_positives=fp,\n                precision=tp/(tp+fp) if tp+fp else None,\n                recall=tp/positives if positives else None,\n                false_positives_per_image=fp/len(gt["images"]),\n                AP50_saved_predictions_101point=ap if iou == .5 else None,\n                geometric_size_definition="s=sqrt(width*height), 배포 이미지 픽셀 기준",\n                geometric_2to20=geometric_range(2,20),\n                geometric_2to8=geometric_range(2,8),\n                geometric_size_recall={"2<=s<8": geometric_range(2,8),\n                                       "8<=s<12": geometric_range(8,12),\n                                       "12<=s<20": geometric_range(12,20)},\n                tiny_le20=dict(positives=tiny_count, hits=tiny_hits,\n                               recall=tiny_hits/tiny_count if tiny_count else None),\n                size_recall={k: dict(positives=counts[k], hits=hits[k],\n                                    recall=hits[k]/counts[k] if counts[k] else None)\n                             for k in ("0<d<=8", "8<d<=12", "12<d<=20", "20<d")})\n', 'workflow.py': '"""Colab 설정·데이터 준비·실행·비교·다운로드를 위한 공통 처리."""\nimport csv\nimport argparse\nimport hashlib\nimport json\nimport shutil\nimport subprocess\nimport os\nimport zipfile\nfrom pathlib import Path\n\n\ndef digest(path):\n    h=hashlib.sha256()\n    with Path(path).open("rb") as stream:\n        for chunk in iter(lambda:stream.read(1024*1024),b""):\n            h.update(chunk)\n    return h.hexdigest()\n\n\ndef dump(path,value):\n    path=Path(path)\n    path.parent.mkdir(parents=True,exist_ok=True)\n    path.write_text(json.dumps(value,ensure_ascii=False,indent=2),encoding="utf-8")\n\n\ndef prepare_dataset(config,work,zip_path):\n    from metrics import merge_splits,load_dataset\n    work=Path(work)\n    signature=digest(zip_path)\n    root=work/"datasets"/config["dataset_name"]/signature[:12]\n    stamp=root/"prepared.json"\n    reused=stamp.is_file() and json.loads(stamp.read_text()).get("zip_sha256")==signature\n    if not reused:\n        root.mkdir(parents=True,exist_ok=True)\n        with zipfile.ZipFile(zip_path) as archive:\n            if sum(m.file_size for m in archive.infolist())+1024**3>shutil.disk_usage(work).free:\n                raise RuntimeError("데이터 압축 해제에 필요한 공간이 부족합니다.")\n            for member in archive.infolist():\n                resolved=(root/member.filename).resolve()\n                if root.resolve()!=resolved and root.resolve() not in resolved.parents:\n                    raise ValueError("ZIP에 데이터 폴더 밖을 가리키는 경로가 있습니다.")\n                if (member.external_attr>>16)&0o170000==0o120000:\n                    raise ValueError("ZIP의 심볼릭 링크는 지원하지 않습니다.")\n            archive.extractall(root)\n        dump(stamp,dict(zip_sha256=signature))\n    split=config["split"]\n    parts=("train","valid","test") if split=="all" else (split,)\n    sources=[]\n    for part in parts:\n        paths=[p for p in root.rglob("_annotations.coco.json") if p.parent.name==part]\n        if len(paths)!=1:\n            raise ValueError(part+" 분할의 정답 JSON을 한 개 찾을 수 없습니다.")\n        sources.append((part,str(paths[0])))\n    annotations=root/("merged_%s.coco.json"%split)\n    merge_splits(sources,annotations)\n    gt,summary=load_dataset(annotations,["person water","person sand"])\n    water,water_summary=load_dataset(annotations,["person water"])\n    if config["dataset_name"]=="AnnotationPiriapolis_v4" and split=="all":\n        if (summary["image_count"],summary["target_count"],water_summary["target_count"])!=(635,8099,2904):\n            raise ValueError("AnnotationPiriapolis v4의 전체 635장·전체 사람 8099개·물속 사람 2904개와 다릅니다.")\n    state=dict(task="dataset",dataset_name=config["dataset_name"],split=split,\n               annotations=str(annotations),zip_sha256=signature,\n               images=summary["image_count"],all_people=summary["target_count"],water_only=water_summary["target_count"])\n    dump(work/"prepared_input.json",state)\n    print("데이터 준비 완료: %d장 / 전체 사람 %d개 / 물속 사람 %d개"%(state["images"],state["all_people"],state["water_only"]),flush=True)\n    print("기존 압축 해제 파일을 재사용했습니다." if reused else "데이터 압축을 해제했습니다.",flush=True)\n    return state\n\n\ndef import_baseline(zip_path,destination):\n    destination=Path(destination)\n    needed=("predictions_coco.json","ground_truth_water.json","run_manifest.json")\n    with zipfile.ZipFile(zip_path) as archive:\n        candidates=[n for n in archive.namelist() if n.endswith("/full/predictions_coco.json")]\n        if len(candidates)!=1:\n            raise ValueError("기존 전체 검사 결과 ZIP의 full/predictions_coco.json을 한 개 찾을 수 없습니다.")\n        prefix=candidates[0].rsplit("/",1)[0]+"/"\n        payload={name:archive.read(prefix+name) for name in needed}\n        manifest=json.loads(payload["run_manifest.json"])\n        if manifest["mode"]!="full":\n            raise ValueError("비교용 ZIP은 전체 검사(full) 결과여야 합니다.")\n    for name,data in payload.items():\n        json.loads(data)\n    destination.mkdir(parents=True,exist_ok=True)\n    for name,data in payload.items():\n        (destination/name).write_bytes(data)\n    return destination\n\n\ndef align_baseline(baseline,annotations,checkpoint,settings,destination):\n    from metrics import load_dataset\n    baseline,destination=Path(baseline),Path(destination)\n    manifest=json.loads((baseline/"run_manifest.json").read_text(encoding="utf-8"))\n    if manifest.get("checkpoint_sha256")!=digest(checkpoint):\n        raise ValueError("기존 전체 검사와 현재 가중치가 달라 같은 모델의 분할 효과를 비교할 수 없습니다.")\n    for key in ("long_edge","short_edge","max_detections","save_min_score"):\n        if manifest.get(key)!=settings.get(key):\n            raise ValueError("기존 전체 검사와 설정이 다릅니다: "+key)\n    gt,_=load_dataset(annotations,["person water"])\n    old=json.loads((baseline/"ground_truth_water.json").read_text(encoding="utf-8"))\n    def key(im):\n        return (im.get("source_split",""),im.get("source_image_id",im["id"]),Path(im["file_name"]).name)\n    old_map={key(im):im["id"] for im in old["images"]}\n    if any(key(im) not in old_map for im in gt["images"]):\n        raise ValueError("기존 전체 검사 결과에 현재 데이터의 이미지가 모두 포함돼 있지 않습니다.")\n    remap={old_map[key(im)]:im["id"] for im in gt["images"]}\n    # 이름이 같은 파일의 주석이 변경된 비교를 막는다.\n    def boxes(raw,ids=None):\n        values=[]\n        for a in raw["annotations"]:\n            image_id=a["image_id"]\n            if ids is not None:\n                if image_id not in ids:continue\n                image_id=ids[image_id]\n            values.append((image_id,tuple(a["bbox"]),bool(a.get("ignore")),bool(a.get("iscrowd"))))\n        return sorted(values)\n    if boxes(old,remap)!=boxes(gt):\n        raise ValueError("기존 전체 검사와 현재 물속 사람 주석이 달라 비교를 중단했습니다.")\n    predictions=json.loads((baseline/"predictions_coco.json").read_text(encoding="utf-8"))\n    predictions=[dict(p,image_id=remap[p["image_id"]]) for p in predictions if p["image_id"] in remap]\n    dump(destination/"predictions_coco.json",predictions)\n    dump(destination/"ground_truth_water.json",gt)\n    dump(destination/"run_manifest.json",dict(manifest,comparison_image_ids=list(remap.values())))\n    return destination\n\n\ndef cached_predictions_valid(out,settings,annotations,limit):\n    from metrics import load_dataset,image_path\n    out=Path(out)\n    try:\n        old=json.loads((out/"run_manifest.json").read_text(encoding="utf-8"))\n        status=json.loads((out/"run_status.json").read_text(encoding="utf-8"))\n        if status["state"]!="완료" or not (out/"predictions_coco.json").is_file():return False\n        keys=("mode","long_edge","short_edge","tile_size","tile_overlap","merge_iou","save_min_score","max_detections")\n        if any(old.get(k)!=settings.get(k) for k in keys):return False\n        if old["checkpoint_sha256"]!=digest(settings["checkpoint"]) or old["annotations_sha256"]!=digest(annotations):return False\n        if old.get("engine_sha256")!=digest(Path(settings["repo"]).parent/"helpers/engine.py"):return False\n        commit=subprocess.check_output(["git","-C",settings["repo"],"rev-parse","HEAD"],text=True).strip()\n        if old.get("source_commit")!=commit:return False\n        gt,_=load_dataset(annotations,["person water"],limit)\n        if old["image_ids"]!=[im["id"] for im in gt["images"]]:return False\n        cache=out/("cache_"+status["cache_key"])\n        return all(json.loads((cache/(str(im["id"])+".json")).read_text(encoding="utf-8"))["image_sha256"]==digest(image_path(annotations,im["file_name"])) for im in gt["images"])\n    except (OSError,ValueError,KeyError):\n        return False\n\n\ndef run_dataset(config,work,store,checkpoint,env):\n    from summarize_scopes import summarize_scopes\n    work,store=Path(work),Path(store)\n    state=json.loads((work/"prepared_input.json").read_text(encoding="utf-8"))\n    if any(state.get(k)!=config[k] for k in ("task","dataset_name","split")):\n        raise ValueError("데이터 선택이 변경됐습니다. 3단계부터 다시 실행하세요.")\n    mode=config["mode"]\n    label="full" if mode=="full" else "tiles_%d_overlap%g"%(config["tile_size"],config["tile_overlap"])\n    out=store/"evaluation"/config["dataset_name"]/config["split"]/label\n    if config.get("weight_source", "public") != "public":\n        out=store/"evaluation_models"/digest(checkpoint)[:16]/config["dataset_name"]/config["split"]/label\n    settings=dict(repo=str(work/"SSPNet"),checkpoint=str(checkpoint),mode=mode,\n                  long_edge=config["long_edge"],short_edge=config["short_edge"],tile_size=config["tile_size"],\n                  tile_overlap=config["tile_overlap"],merge_iou=.5,report_score=config["report_score"],\n                  save_min_score=.001,max_detections=500,target_names=["person water"],\n                  dataset_name=config["dataset_name"],split=config["split"],match_iou=config["match_iou"])\n    dump(work/"evaluation_settings.json",settings)\n    python=work/"venv/bin/python"\n    if cached_predictions_valid(out,settings,state["annotations"],config["max_images"]):\n        print("동일한 추론 결과를 재사용하고 점수만 다시 집계합니다.",flush=True)\n    else:\n        subprocess.run([str(python),str(work/"helpers/engine.py"),"--settings",str(work/"evaluation_settings.json"),\n                        "--annotations",state["annotations"],"--limit",str(config["max_images"]),"--out",str(out)],env=env,check=True)\n    summarize_scopes(state["annotations"],out,config["report_score"],config["match_iou"])\n    outputs=[out]\n    baseline=Path(config.get("baseline_path") or store/"baselines"/config["dataset_name"])\n    if config.get("compare_with_full") and mode!="full":\n        if not (baseline/"predictions_coco.json").is_file():\n            baseline=store/"evaluation"/config["dataset_name"]/config["split"]/"full"\n        if not (baseline/"predictions_coco.json").is_file():\n            raise ValueError("기존 전체 검사 결과가 없습니다. 3단계에서 비교용 ZIP을 준비하세요.")\n        destination=out/"baseline_full_rescored"\n        align_baseline(baseline,state["annotations"],checkpoint,settings,destination)\n        # 부분 평가에서는 양쪽 모두 현재 처리한 이미지로만 비교한다.\n        current=json.loads((out/"ground_truth_water.json").read_text(encoding="utf-8"))\n        aligned=json.loads((destination/"ground_truth_water.json").read_text(encoding="utf-8"))\n        ids={im["id"] for im in current["images"]}\n        aligned["images"]=[im for im in aligned["images"] if im["id"] in ids]\n        aligned["annotations"]=[a for a in aligned["annotations"] if a["image_id"] in ids]\n        dump(destination/"ground_truth_water.json",aligned)\n        summarize_scopes(state["annotations"],destination,config["report_score"],config["match_iou"])\n        rows=[]\n        for scope in ("all_people","water_only"):\n            for method,folder in (("전체 검사",destination),("분할 검사",out)):\n                m=json.loads((folder/("metrics_"+scope+".json")).read_text(encoding="utf-8"))\n                rows.append(dict(평가범위=m["scope_label"],방식=method,이미지=m["images"],정답=m["positives"],정탐=m["true_positives"],\n                                 재현율=m["recall"],정밀도=m["precision"],오탐=m["false_positives"],이미지당오탐=m["false_positives_per_image"],\n                                 작은사람재현율=m["geometric_2to20"]["recall"],최소구간재현율=m["geometric_2to8"]["recall"]))\n        dump(out/"전체검사_분할검사_비교.json",rows)\n        with (out/"전체검사_분할검사_비교.csv").open("w",encoding="utf-8-sig",newline="") as stream:\n            writer=csv.DictWriter(stream,fieldnames=list(rows[0]));writer.writeheader();writer.writerows(rows)\n        print("전체 검사·분할 검사 비교표를 같은 점수·IoU·이미지·정답 범위로 저장했습니다.",flush=True)\n    dump(work/"last_output.json",dict(paths=[str(p) for p in outputs],task="dataset",label=label,config=config))\n    return out\n\n\nif __name__=="__main__":\n    parser=argparse.ArgumentParser(description="SSPNet 통합 평가 처리")\n    parser.add_argument("action",choices=["prepare","baseline","run"])\n    parser.add_argument("--config",required=True)\n    parser.add_argument("--zip")\n    parser.add_argument("--work",default="/content/sspnet_free")\n    parser.add_argument("--store",required=True)\n    args=parser.parse_args()\n    config=json.loads(Path(args.config).read_text(encoding="utf-8"))\n    if args.action=="prepare":\n        prepare_dataset(config,args.work,args.zip)\n    elif args.action=="baseline":\n        import_baseline(args.zip,Path(args.store)/"baselines"/config["dataset_name"])\n        print("기존 전체 검사 결과를 준비했습니다. 다시 GPU로 추론하지 않습니다.",flush=True)\n    else:\n        run_dataset(config,args.work,args.store,Path(config.get("checkpoint") or Path(args.work)/"faster_rcnn_sspnet.pth"),os.environ.copy())\n', 'summarize_scopes.py': '"""한 번 저장한 person 예측을 전체 사람·물속 사람 기준으로 각각 집계한다."""\nimport argparse\nimport json\nfrom collections import defaultdict\nfrom pathlib import Path\n\nfrom metrics import geometric_size, load_dataset, match_predictions\n\n\ndef score_scope(gt, predictions, score, iou):\n    selected = [p for p in predictions if p["score"] >= score]\n    matches = match_predictions(gt, selected, iou)\n    normal = [a for a in gt["annotations"] if not a.get("ignore") and not a.get("iscrowd")]\n    annotations_by_image = defaultdict(list)\n    matches_by_image = defaultdict(list)\n    for annotation in normal:\n        annotations_by_image[annotation["image_id"]].append(annotation)\n    for record in matches:\n        matches_by_image[record[0]["image_id"]].append(record)\n\n    def group(images):\n        annotations = [a for im in images for a in annotations_by_image[im["id"]]]\n        records = [r for im in images for r in matches_by_image[im["id"]]]\n        hits = [a for _, a, ignored in records if a is not None and not ignored]\n        false_positive = sum(a is None and not ignored for _, a, ignored in records)\n        true_positive = len(hits)\n        def sizes(low, high):\n            positives = sum(low <= geometric_size(a["bbox"]) < high for a in annotations)\n            found = sum(low <= geometric_size(a["bbox"]) < high for a in hits)\n            return dict(positives=positives, hits=found, recall=found/positives if positives else None)\n        return dict(images=len(images), positives=len(annotations), true_positives=true_positive,\n                    false_positives=false_positive, false_negatives=len(annotations)-true_positive,\n                    precision=true_positive/(true_positive+false_positive) if true_positive+false_positive else None,\n                    recall=true_positive/len(annotations) if annotations else None,\n                    false_positives_per_image=false_positive/len(images),\n                    geometric_2to20=sizes(2,20), geometric_2to8=sizes(2,8))\n\n    result = group(gt["images"])\n    result.update(score_threshold=score, iou=iou,\n                  geometric_size_definition="s=sqrt(width*height), 배포 이미지 픽셀 기준")\n    splits = sorted({im.get("source_split", "single") for im in gt["images"]})\n    result["by_split"] = {s:group([im for im in gt["images"] if im.get("source_split", "single")==s]) for s in splits}\n    return result\n\n\ndef summarize_scopes(annotations, out, score=.25, iou=.3):\n    if not 0 < score <= 1 or not 0 < iou <= 1:\n        raise ValueError("점수와 IoU 기준은 0 초과 1 이하여야 합니다.")\n    out = Path(out)\n    predictions = json.loads((out/"predictions_coco.json").read_text(encoding="utf-8"))\n    saved_path = out/"ground_truth_water.json"\n    image_ids = {im["id"] for im in json.loads(saved_path.read_text(encoding="utf-8"))["images"]} if saved_path.is_file() else None\n    scopes = (("all_people", ["person water","person sand"], 8099, "전체 사람"),\n              ("water_only", ["person water"], 2904, "물속 사람"))\n    values = {}\n    reference = {}\n    thresholds = [iou] if iou==.5 else [iou,.5]\n    for name, classes, expected, label in scopes:\n        gt, _ = load_dataset(annotations, classes)\n        available_images = len(gt["images"])\n        if image_ids is not None:\n            gt["images"] = [im for im in gt["images"] if im["id"] in image_ids]\n            gt["annotations"] = [a for a in gt["annotations"] if a["image_id"] in image_ids]\n        if not gt["images"]:\n            raise ValueError("평가할 이미지가 없습니다.")\n        gt["info"]["description"] = label+" 평가용 정답"\n        gt["categories"] = [dict(id=1,name="person" if name=="all_people" else "person water")]\n        if name=="all_people":\n            (out/"ground_truth_all_people.json").write_text(json.dumps(gt,ensure_ascii=False),encoding="utf-8")\n        for threshold in thresholds:\n            result = score_scope(gt, predictions, score, threshold)\n            result.update(target_classes=classes,scope_label=label,\n                          partial=len(gt["images"])<available_images,available_image_count=available_images)\n            primary = threshold==iou\n            filename = "metrics_%s%s.json" % (name,"" if primary else "_iou05_reference")\n            (out/filename).write_text(json.dumps(result,ensure_ascii=False,indent=2),encoding="utf-8")\n            (values if primary else reference)[name] = result\n\n    def percent(value):\n        return "계산 불가" if value is None else "%.1f%%" % (100*value)\n    lines = ["SSPNet 전체 사람·물속 사람 동시 평가", "", "동일한 person 예측을 두 가지 정답 범위로 각각 평가했습니다.",\n             "이미지: %d장 / 예측 점수: %.2f / 주 결과 IoU: %.2f" % (values["all_people"]["images"],score,iou), ""]\n    for name, _, _, label in scopes:\n        m = values[name]\n        lines += ["[%s 기준]" % label,\n                  "정답 %s개 / 정탐 %s개 / 놓친 정답 %s개" % tuple(format(m[k],",") for k in ("positives","true_positives","false_negatives")),\n                  "재현율 %s / 정밀도 %s" % (percent(m["recall"]),percent(m["precision"])),\n                  "주석 기준 오탐 %s개 / 이미지당 %.2f개" % (format(m["false_positives"],","),m["false_positives_per_image"]), ""]\n    lines += ["전체 사람 기준에서는 모래사장 사람도 정답입니다.",\n              "물속 사람 기준에서는 모래사장 사람에 연결되는 예측도 오탐에 포함됩니다.",\n              "주석 누락·위치 오류는 보정하지 않았습니다. 동일한 예측을 범위별로 다시 매칭하므로 정탐 수는 단순 합산으로 연결되지 않을 수 있습니다.",\n              "크기 제한 없는 결과가 주 결과이며 크기별 재현율과 train·valid·test별 지표도 각 JSON에 저장했습니다.",\n              "IoU 0.5 참고 결과도 저장했습니다." if reference else "현재 IoU 0.5로 집계했습니다."]\n    report = "\\n".join(lines)+"\\n"\n    (out/"전체사람_물속사람_결과요약.txt").write_text(report,encoding="utf-8")\n    print(report,flush=True)\n    return values\n\n\nif __name__=="__main__":\n    parser=argparse.ArgumentParser(description="전체 사람과 물속 사람의 동시 결과 집계")\n    parser.add_argument("--annotations",required=True)\n    parser.add_argument("--out",required=True)\n    parser.add_argument("--score",type=float,default=.25)\n    parser.add_argument("--iou",type=float,default=.3)\n    args=parser.parse_args()\n    summarize_scopes(args.annotations,args.out,args.score,args.iou)\n', 'train.py': '"""이전 실험 재현용: 검증 없이 YoloBeachTrain 전체로 학습하던 코드. 새 학습에는 train.py를 사용한다."""\nimport argparse\nimport hashlib\nimport json\nfrom pathlib import Path\n\n\ndef digest(path):\n    h = hashlib.sha256()\n    with open(path, \'rb\') as f:\n        for chunk in iter(lambda: f.read(1024 * 1024), b\'\'):\n            h.update(chunk)\n    return h.hexdigest()\n\n\ndef make_annotations(source, dest):\n    raw = json.loads(Path(source).read_text(encoding=\'utf-8\'))\n    names = {c[\'id\']: c[\'name\'].lower().replace(\'_\', \' \').strip() for c in raw[\'categories\']}\n    wanted = {i for i, name in names.items() if name in (\'person water\', \'person sand\')}\n    if set(names[i] for i in wanted) != {\'person water\', \'person sand\'}:\n        raise ValueError(\'두 사람 라벨을 찾지 못했습니다: \' + repr(names))\n    raw[\'images\'] = [dict(im, corner=im.get(\'corner\', [0, 0, im[\'width\'], im[\'height\']])) for im in raw[\'images\']]\n    raw[\'annotations\'] = [dict(a, category_id=1, ignore=int(bool(a.get(\'ignore\', False) or a.get(\'iscrowd\', False))),\n                               uncertain=int(bool(a.get(\'uncertain\', False))), segmentation=a.get(\'segmentation\', []))\n                          for a in raw[\'annotations\'] if a[\'category_id\'] in wanted]\n    raw[\'categories\'] = [dict(id=1, name=\'person\', supercategory=\'person\')]\n    Path(dest).write_text(json.dumps(raw, ensure_ascii=False), encoding=\'utf-8\')\n    return len(raw[\'images\']), len(raw[\'annotations\'])\n\n\ndef main():\n    p = argparse.ArgumentParser()\n    for arg in (\'repo\', \'annotations\', \'checkpoint\', \'output\', \'mode\'):\n        p.add_argument(\'--\' + arg, required=True)\n    p.add_argument(\'--epochs\', type=int, default=5)\n    p.add_argument(\'--smoke-steps\', type=int, default=50)\n    p.add_argument(\'--resume\', action=\'store_true\')\n    a = p.parse_args()\n    # 원본 ICNegSampler는 정답이 없는 이미지에서 빈 행렬의 max를 호출한다.\n    # 이 경우 모든 후보의 정답과의 겹침은 0이다.\n    sampler = Path(a.repo) / \'mmdet/core/bbox/samplers/ic_base_sampler.py\'\n    sampler_text = sampler.read_text(encoding=\'utf-8\')\n    old = \'        iod_max_overlaps, _ = iod_overlaps.max(dim=0)\'\n    new = \'        iod_max_overlaps = iod_overlaps.max(dim=0)[0] if gt_bboxes.numel() else bboxes.new_zeros((bboxes.shape[0],))\'\n    if old in sampler_text:\n        sampler.write_text(sampler_text.replace(old, new), encoding=\'utf-8\')\n    elif new not in sampler_text:\n        raise RuntimeError(\'ICNegSampler 코드가 예상과 다릅니다.\')\n    heatmap = Path(a.repo) / \'mmdet/models/module/heatmap.py\'\n    heatmap_text = heatmap.read_text(encoding=\'utf-8\')\n    old_threshold = \'threshold = -bg_score_sorted[bg_num - 1]\'\n    new_threshold = \'threshold = -bg_score_sorted[min(bg_num, len(bg_score_sorted)) - 1] if bg_num > 0 and len(bg_score_sorted) else float("inf")\'\n    if heatmap_text.count(old_threshold) == 2:\n        heatmap.write_text(heatmap_text.replace(old_threshold, new_threshold), encoding=\'utf-8\')\n    elif heatmap_text.count(new_threshold) != 2:\n        raise RuntimeError(\'Heatmap 코드가 예상과 다릅니다.\')\n    import torch\n    from mmcv import Config\n    from mmdet.apis import set_random_seed, train_detector\n    from mmdet.datasets import build_dataset\n    from mmdet.models import build_detector\n\n    if not torch.cuda.is_available():\n        raise RuntimeError(\'GPU 런타임이 필요합니다.\')\n    output = Path(a.output)\n    output.mkdir(parents=True, exist_ok=True)\n    ann = Path(a.annotations).with_name(\'training_person.coco.json\')\n    counts = make_annotations(a.annotations, ann)\n    if counts[0] != 2961:\n        raise ValueError(\'YoloBeachTrain v2 전체 2961장이 아닙니다: \' + str(counts))\n    cfg = Config.fromfile(str(Path(a.repo) / \'configs/sspnet/faster_rcnn_r50_sspnet_1x_coco.py\'))\n    cfg.model.pretrained = None\n    pipeline = [dict(type=\'LoadImageFromFile\'), dict(type=\'LoadAnnotations\', with_bbox=True),\n                dict(type=\'Resize\', img_scale=(1280, 1024), keep_ratio=True),\n                dict(type=\'RandomFlip\', flip_ratio=0.0),\n                dict(type=\'Normalize\', mean=[123.675,116.28,103.53], std=[58.395,57.12,57.375], to_rgb=True),\n                dict(type=\'Pad\', size_divisor=32), dict(type=\'DefaultFormatBundle\'),\n                dict(type=\'Collect\', keys=[\'img\',\'gt_bboxes\',\'gt_labels\',\'gt_bboxes_ignore\'])]\n    cfg.data = dict(samples_per_gpu=1, workers_per_gpu=1,\n                    train=dict(type=\'CocoDataset\', ann_file=str(ann), img_prefix=str(ann.parent) + \'/\',\n                               classes=(\'person\',), filter_empty_gt=False, pipeline=pipeline))\n    cfg.optimizer = dict(type=\'SGD\', lr=0.0001, momentum=0.9, weight_decay=0.0001)\n    cfg.optimizer_config = dict(grad_clip=None)\n    cfg.lr_config = dict(policy=\'fixed\')\n    cfg.workflow = [(\'train\', 1)]\n    cfg.gpu_ids = [0]\n    cfg.seed = 0\n    cfg.work_dir = str(output)\n    cfg.load_from = None\n    cfg.resume_from = None\n    cfg.log_config = dict(interval=10, hooks=[dict(type=\'TextLoggerHook\')])\n    cfg.runner = dict(type=\'IterBasedRunner\', max_iters=a.smoke_steps) if a.mode == \'smoke\' else dict(type=\'EpochBasedRunner\', max_epochs=a.epochs)\n    cfg.pop(\'total_epochs\', None)\n    cfg.checkpoint_config = dict(interval=a.smoke_steps if a.mode == \'smoke\' else 1, by_epoch=a.mode != \'smoke\', max_keep_ckpts=2)\n    signature = dict(dataset_sha256=digest(a.annotations), initial_sha256=digest(a.checkpoint),\n                     mode=a.mode, epochs=a.epochs, smoke_steps=a.smoke_steps,\n                     learning_rate=0.0001, batch_size=1, image_scale=[1280,1024], seed=0)\n    signature_path = output / \'experiment.json\'\n    if signature_path.exists() and json.loads(signature_path.read_text(encoding=\'utf-8\')) != signature:\n        raise ValueError(\'기존 실행의 데이터 또는 설정이 다릅니다. 새 실행 폴더를 사용하세요.\')\n    latest = output / \'latest.pth\'\n    if a.resume:\n        if not signature_path.is_file() or not latest.is_file():\n            raise ValueError(\'재개할 설정 또는 latest.pth가 없습니다.\')\n        cfg.resume_from = str(latest)\n    elif signature_path.exists() or latest.exists():\n        raise ValueError(\'기존 실행 폴더입니다. 재개하거나 새 실행 폴더를 지정하세요.\')\n    signature_path.write_text(json.dumps(signature, ensure_ascii=False, indent=2), encoding=\'utf-8\')\n    cfg.dump(str(output / \'resolved_config.py\'))\n    set_random_seed(0, deterministic=True)\n    dataset = build_dataset(cfg.data.train)\n    print(\'학습 이미지/사람 주석:\', counts, \'실제 데이터셋:\', len(dataset), flush=True)\n    if len(dataset) != counts[0]:\n        raise ValueError(\'데이터셋에서 제외된 이미지가 있습니다.\')\n    model = build_detector(cfg.model, train_cfg=cfg.train_cfg, test_cfg=cfg.test_cfg)\n    checkpoint = torch.load(a.checkpoint, map_location=\'cpu\')\n    state = checkpoint.get(\'state_dict\', checkpoint)\n    if all(k.startswith(\'module.\') for k in state):\n        state = {k[7:]: v for k, v in state.items()}\n    model.load_state_dict(state, strict=True)\n    model.CLASSES = (\'person\',)\n    train_detector(model, dataset, cfg, distributed=False, validate=False)\n    if a.mode == \'train\':\n        final = output / (\'epoch_%d.pth\' % a.epochs)\n        if not final.is_file():\n            raise RuntimeError(\'마지막 epoch 가중치가 없습니다.\')\n        print(\'최종 학습 가중치:\', final, flush=True)\n\n\nif __name__ == \'__main__\':\n    main()\n'}
from google.colab import drive
drive.mount('/content/drive')
STORE=Path('/content/drive/MyDrive/SSPNet_free_colab')
STORE.mkdir(parents=True,exist_ok=True)
HELPERS=WORK/'helpers'
HELPERS.mkdir(exist_ok=True)
for name,source in HELPER_SOURCES.items(): (HELPERS/name).write_text(source,encoding='utf-8')
subprocess.run([sys.executable,str(HELPERS/'setup_environment.py')],check=True)
PYTHON=WORK/'venv/bin/python'
REPO=WORK/'SSPNet'
ENV=os.environ.copy()
ENV.pop('PYTHONHOME',None)
ENV.update(PYTHONNOUSERSITE='1',PYTHONPATH=str(REPO)+os.pathsep+str(HELPERS),MPLBACKEND='Agg')
def checked(args):
    process=subprocess.Popen([str(x) for x in args],env=ENV,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
    for line in process.stdout: print(line,end='',flush=True)
    code=process.wait()
    if code: raise subprocess.CalledProcessError(code,args)
CHECKPOINT=STORE/'weights/faster_rcnn_sspnet.pth'
CHECKPOINT.parent.mkdir(parents=True,exist_ok=True)
if not CHECKPOINT.is_file():
    checked([PYTHON,'-c',"import gdown; p=gdown.download(id='1IfPCt5xZqqBJ3sYVIuD5F9l29Jcy2Hn1',output=%r,quiet=False); assert p"%str(CHECKPOINT)])
import hashlib
def filehash(path):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for block in iter(lambda:f.read(1024*1024),b''): h.update(block)
    return h.hexdigest()
if filehash(CHECKPOINT) != '912cc5b6f8a45b0f4521c9bf325868ed10dd4757497fb6ba8843fdbd26b09e4a':
    raise ValueError('공개 가중치 해시가 기준과 다릅니다. 원본 가중치를 확인하세요.')
print('환경과 공개 가중치 준비 완료')


In [ ]:
#@title 3. 학습 데이터 준비
DATA_ZIP=Path(DATASET_ZIP_PATH) if DATASET_ZIP_PATH else WORK/'YoloBeachTrain_v2.zip'
if not DATA_ZIP.is_file():
    from google.colab import files
    print('YoloBeachTrain.v2i.coco.zip을 선택하세요.')
    uploaded=files.upload()
    if len(uploaded)!=1: raise ValueError('ZIP 한 개를 선택하세요.')
    DATA_ZIP.write_bytes(next(iter(uploaded.values())))
CONFIG=dict(task='dataset',dataset_name='YoloBeachTrain_v2',split='all')
config_path=WORK/'training_data_config.json'
config_path.write_text(json.dumps(CONFIG),encoding='utf-8')
checked([PYTHON,HELPERS/'workflow.py','prepare','--config',config_path,'--zip',DATA_ZIP,'--store',STORE])
prepared=json.loads((WORK/'prepared_input.json').read_text(encoding='utf-8'))
ANNOTATIONS=Path(prepared['annotations'])
print('학습 데이터 준비:',prepared['images'],'장')


## 4. 시험 학습

50회 업데이트가 기본입니다. 오류·소요 시간·GPU 메모리를 확인합니다. 시험 결과로 Piriapolis를 평가하지 않습니다. 다시 시험하려면 실행 이름을 바꾸세요.


In [ ]:
#@title 4. 시험 학습
SMOKE_DIR=STORE/'training'/RUN_NAME/'smoke'
checked([PYTHON,HELPERS/'train.py','--repo',REPO,'--annotations',ANNOTATIONS,'--checkpoint',CHECKPOINT,
         '--output',SMOKE_DIR,'--mode','smoke','--epochs',EPOCHS,'--smoke-steps',SMOKE_STEPS])
print('시험 학습 완료. 본 학습 횟수를 확정한 뒤 5단계를 실행하세요.')


## 5. 본 학습과 최종 가중치

시험 학습 후 횟수를 확정하고 실행합니다. 재개할 때는 설정을 바꾸지 마세요. 본 학습이 끝나면 표시되는 `epoch_N.pth` 경로를 평가 노트북의 `trained_path`에 입력하고 `full`을 선택합니다. Drive가 연결되어 있어야 합니다.


In [ ]:
#@title 5. 본 학습
RESUME = False #@param {type:"boolean"}
if not (STORE/'training'/RUN_NAME/'smoke'/('iter_%d.pth'%SMOKE_STEPS)).is_file():
    raise RuntimeError('같은 실행 이름과 설정으로 4단계 시험 학습을 먼저 완료하세요.')
TRAIN_DIR=STORE/'training'/RUN_NAME/'main'
args=[PYTHON,HELPERS/'train.py','--repo',REPO,'--annotations',ANNOTATIONS,'--checkpoint',CHECKPOINT,
      '--output',TRAIN_DIR,'--mode','train','--epochs',EPOCHS,'--smoke-steps',SMOKE_STEPS]
if RESUME: args.append('--resume')
checked(args)
print('평가 노트북에 입력할 최종 가중치 경로:',TRAIN_DIR/('epoch_%d.pth'%EPOCHS))
